# 00 — Data Pre-Processing

**Purpose:** Load the raw Qualtrics export, clean it, and save a tidy processed dataset ready for analysis.

**Input:** `data/raw/Medical Provider Dental Trauma Assessment_April 9, 2026_20.07.csv`  
**Output:** `data/processed/survey_clean.csv`

### Survey structure overview
- **Self-assessment:** Knowledge/confidence ratings (0–10) before seeing cases
- **Case 1 — Primary tooth avulsion:** Q1, Q2, Q3
- **Case 2 — Permanent tooth avulsion:** Q4, Q39, Q41, Q40, Q42
- **Case 3 — Crown fracture:** Q44, Q45, Q46, Q47
- **Score:** `SC0` — total correct out of 12
- **Randomization:** `random` — 1 = Control (PDF), 2 = Intervention (ChatGPT)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load raw data

Qualtrics exports have **3 header rows**:
- Row 0: short variable names (used as column headers)
- Row 1: full question text
- Row 2: Qualtrics import IDs

We keep row 0 as the header and drop rows 1–2.

In [ ]:
RAW_PATH = Path("../data/raw/Medical Provider Dental Trauma Assessment_April 9, 2026_20.07.csv")

df_raw = pd.read_csv(RAW_PATH, header=0, skiprows=[1, 2], low_memory=False)

print(f"Shape: {df_raw.shape}")
df_raw.head(3)

## 2. Filter to complete responses

Keep only rows where `Finished == True` and `Progress == 100`.

In [ ]:
df = df_raw.copy()

print(f"Total responses before filtering: {len(df)}")
print(f"  Finished=True:   {df['Finished'].eq('True').sum()}")
print(f"  Finished=False:  {df['Finished'].eq('False').sum()}")

df = df[df["Finished"].eq("True") & df["Progress"].astype(int).eq(100)].copy()
df = df.reset_index(drop=True)

print(f"\nResponses after filtering: {len(df)}")

## 3. Drop metadata / PII columns

In [ ]:
drop_cols = [
    "StartDate", "EndDate", "Status", "IPAddress",
    "Progress", "Finished", "RecordedDate",
    "RecipientLastName", "RecipientFirstName", "RecipientEmail",
    "ExternalReference", "LocationLatitude", "LocationLongitude",
    "DistributionChannel", "UserLanguage",
    # Qualtrics consent checkbox columns (no analytic value)
    "Q55", "Q56", "Q54",
    "block2progress",
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print("Remaining columns:", df.columns.tolist())

## 4. Rename columns to readable names

In [ ]:
rename_map = {
    "ResponseId":               "response_id",
    "Duration (in seconds)":    "duration_sec",
    # Self-assessment
    "Q1_1":  "self_knowledge_tdi",
    "Q2_1":  "self_confidence_avulsion",
    "Q3_1":  "self_confidence_fracture",
    # Demographics
    "Q5":                    "edu_status",
    "Q5_4_TEXT":             "edu_status_other",
    "Medical Student":       "ms_year",
    "Resident/Fellow":       "res_specialty",
    "Resident/Fellow_4_TEXT": "res_specialty_other",
    "Resident/Fellow_5_TEXT": "res_pgy_other",    # second Resident/Fellow col handled below
    "Attending":             "att_specialty",      # first Attending col
    # second Attending col (years since residency) handled below
    # Knowledge questions — Case 1 (primary avulsion)
    "Q1":  "c1_injury_type",
    "Q2":  "c1_treatment",
    "Q3":  "c1_antibiotics",
    # Knowledge questions — Case 2 (permanent avulsion)
    "Q4":  "c2_injury_type",
    "Q39": "c2_treatment",
    "Q41": "c2_tf_60min",
    "Q40": "c2_storage_rank",
    "Q42": "c2_antibiotics",
    # Knowledge questions — Case 3 (crown fracture)
    "Q44": "c3_injury_type",
    "Q45": "c3_treatment",
    "Q46": "c3_imaging",
    "Q47": "c3_antibiotics",
    # Outcome
    "SC0":    "total_score",
    "random": "group",
}

# Handle duplicate column names that Qualtrics creates for branched questions
cols = df.columns.tolist()
res_fellow_indices  = [i for i, c in enumerate(cols) if c == "Resident/Fellow"]
attending_indices   = [i for i, c in enumerate(cols) if c == "Attending"]

if len(res_fellow_indices) == 2:
    cols[res_fellow_indices[1]] = "res_pgy"
if len(attending_indices) == 2:
    cols[attending_indices[1]] = "att_years_since_residency"

df.columns = cols
df = df.rename(columns=rename_map)

print(df.columns.tolist())

## 5. Fix data types

In [ ]:
df["duration_sec"]   = pd.to_numeric(df["duration_sec"], errors="coerce")
df["total_score"]    = pd.to_numeric(df["total_score"], errors="coerce")
df["group"]          = pd.to_numeric(df["group"], errors="coerce").astype("Int64")

self_rating_cols = ["self_knowledge_tdi", "self_confidence_avulsion", "self_confidence_fracture"]
df[self_rating_cols] = df[self_rating_cols].apply(pd.to_numeric, errors="coerce")

# Label the randomization groups
df["group_label"] = df["group"].map({1: "Control (PDF)", 2: "Intervention (ChatGPT)"})

df.dtypes

## 6. Consolidate demographic columns

Qualtrics branches demographic questions by role, so specialty and training level are spread across multiple columns. We unify them into single columns.

In [ ]:
def consolidate_specialty(row):
    """Return a single specialty string regardless of respondent role."""
    if pd.notna(row.get("att_specialty")) and str(row["att_specialty"]).strip():
        return str(row["att_specialty"]).strip()
    if pd.notna(row.get("res_specialty")) and str(row["res_specialty"]).strip():
        spec = str(row["res_specialty"]).strip()
        if spec == "Other (please specify)" and pd.notna(row.get("res_specialty_other")):
            return str(row["res_specialty_other"]).strip()
        return spec
    return np.nan


def consolidate_training_level(row):
    """Return a unified training / experience level string."""
    status = str(row.get("edu_status", "")).strip()
    if status == "Medical Student":
        return str(row.get("ms_year", "")).strip() or np.nan
    if status == "Resident/Fellow":
        pgy = str(row.get("res_pgy", "")).strip()
        if pgy == "Other (please specify)" and pd.notna(row.get("res_pgy_other")):
            return str(row["res_pgy_other"]).strip()
        return pgy or np.nan
    if status == "Attending":
        return str(row.get("att_years_since_residency", "")).strip() or np.nan
    return np.nan


df["specialty"]      = df.apply(consolidate_specialty, axis=1)
df["training_level"] = df.apply(consolidate_training_level, axis=1)

# Clean up edu_status for 'Other' entries
df["edu_status"] = df.apply(
    lambda r: str(r["edu_status_other"]).strip()
    if str(r.get("edu_status", "")).strip() == "Other (please specify)"
    and pd.notna(r.get("edu_status_other"))
    else str(r["edu_status"]).strip(),
    axis=1,
)

# Drop the now-redundant branched columns
branch_cols = [
    "edu_status_other", "ms_year",
    "res_specialty", "res_specialty_other", "res_pgy", "res_pgy_other",
    "att_specialty", "att_years_since_residency",
]
df = df.drop(columns=[c for c in branch_cols if c in df.columns])

df[["edu_status", "specialty", "training_level"]].value_counts(dropna=False)

## 7. Score individual questions (correct / incorrect)

Each response is compared against the answer key and encoded as 1 (correct) or 0 (incorrect).

In [ ]:
ANSWER_KEY = {
    "c1_injury_type":  "Primary tooth avulsion",
    "c1_treatment":    "No treatment needed",
    "c1_antibiotics":  "Antibiotics are not indicated",
    "c2_injury_type":  "Avulsion of a permanent tooth",
    "c2_treatment":    "Gently rinse the root with saline. Re-implant and splint the tooth to the neighboring teeth",
    "c2_tf_60min":     "False",
    "c2_storage_rank": "Milk>Saline>Tap Water>Air",
    # Q42 accepts either amoxicillin or doxycycline
    "c2_antibiotics":  ["Amoxicillin BID for 7 days", "Doxycycline BID for 7 days",
                        "Either amoxicillin or doxycycline BID for 7 days"],
    "c3_injury_type":  "Complicated crown fracture of a permanent incisor",
    "c3_treatment":    "The patient should be sent to a dentist as soon as possible to cover the exposed nerve",
    "c3_imaging":      "Chest and/or soft tissues of the face",
    "c3_antibiotics":  "The patient has an associated tooth root fracture",
}


def score_answer(response, correct):
    if pd.isna(response):
        return np.nan
    r = str(response).strip()
    if isinstance(correct, list):
        return int(any(r.lower() == c.lower() for c in correct))
    return int(r.lower() == str(correct).strip().lower())


for col, answer in ANSWER_KEY.items():
    df[f"{col}_correct"] = df[col].apply(lambda r: score_answer(r, answer))

correct_cols = [c for c in df.columns if c.endswith("_correct")]
df[correct_cols].describe()

## 8. Final overview

In [ ]:
print(f"Final dataset shape: {df.shape}")
print(f"\nGroup distribution:")
print(df["group_label"].value_counts())
print(f"\nEducational status:")
print(df["edu_status"].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df.head()

## 9. Save processed data

In [ ]:
OUT_PATH = Path("../data/processed/survey_clean.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUT_PATH}")